In [ ]:
!pip -q install numpy pandas scipy


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, subprocess, sys
uploaded = files.upload()
archives = [x for x in uploaded if x.endswith(".zip")]
assert archives, "Upload the TNAM_GDSC2_Evaluation_Harness.zip file"
with zipfile.ZipFile(archives[0]) as zf:
    zf.extractall("/content/")
ROOT = Path("/content/TNAM_GDSC2_Evaluation_Harness")
assert (ROOT / "evaluate.py").is_file(), "Harness ZIP not found in expected structure"
print("Loaded harness:", ROOT)


In [ ]:
pred_dir = ROOT / "predictions"
pred_dir.mkdir(exist_ok=True)
uploaded_preds = files.upload()
for name, content in uploaded_preds.items():
    if name.lower().endswith(".csv"):
        (pred_dir / Path(name).name).write_bytes(content)
    elif name.lower().endswith(".zip"):
        upload_zip = Path("/content") / name
        with zipfile.ZipFile(upload_zip) as zf:
            for item in zf.infolist():
                if item.filename.lower().endswith(".csv") and not item.is_dir():
                    (pred_dir / Path(item.filename).name).write_bytes(zf.read(item))
print("Prediction files:", [x.name for x in pred_dir.glob("*.csv")])


In [ ]:
# Natasha corrected sex labels are already included in this package.
sex_file = ROOT / "corrected_sex_labels.csv"
assert sex_file.is_file(), "Natasha corrected labels missing from ZIP"
print("Using mapped corrected sex labels:", sex_file)


In [ ]:
cmd = [sys.executable, str(ROOT / "evaluate.py"),
       "--splits-dir", str(ROOT / "splits"),
       "--predictions-dir", str(ROOT / "predictions"),
       "--output-dir", str(ROOT / "results"),
       "--bootstrap", "1000", "--permutations", "2000", "--seed", "42"]
sex_file = ROOT / "corrected_sex_labels.csv"
if sex_file.is_file():
    cmd += ["--corrected-sex-labels", str(sex_file)]
result = subprocess.run(cmd, check=False, text=True, capture_output=True)
print(result.stdout)
if result.stderr: print(result.stderr)
assert result.returncode == 0, "Evaluation had an error. Check the messages above."


In [ ]:
import pandas as pd
display(pd.read_csv(ROOT / "results" / "h2_performance_gap_table.csv"))
shutil.make_archive("/content/TNAM_H2_results", "zip", ROOT / "results")
files.download("/content/TNAM_H2_results.zip")
